# Sample 4.10 — від baseline до CNN на синтетичних зображеннях

Цей notebook є **безпечним офлайн-прикладом**. Він сам генерує synthetic 16×16 grayscale images трьох класів: vertical stripe, horizontal stripe, cross. Зовнішні datasets та Інтернет не потрібні.

Мета прикладу — показати experiment protocol: fixed seed → train/validation/test → simple baseline → CNN → early stopping → final test → confusion matrix → error analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)

## 1. Генеруємо synthetic dataset

Кожне зображення має форму `(16, 16, 1)`. До базового pattern додаються невеликий random shift і шум. Це дозволяє отримати просту, але не абсолютно тривіальну classification task.

In [ ]:
rng = np.random.default_rng(SEED)
SIZE = 16
N = 1200

def make_image(label):
    img = np.zeros((SIZE, SIZE), dtype=np.float32)
    shift = int(rng.integers(-2, 3))
    center = np.clip(SIZE // 2 + shift, 3, SIZE - 4)
    if label == 0:  # vertical stripe
        img[:, center-1:center+2] = 1.0
    elif label == 1:  # horizontal stripe
        img[center-1:center+2, :] = 1.0
    else:  # cross
        img[:, center-1:center+2] = 1.0
        img[center-1:center+2, :] = 1.0
    img += rng.normal(0, 0.18, img.shape).astype(np.float32)
    return np.clip(img, 0.0, 1.0)

y = rng.integers(0, 3, size=N)
X = np.stack([make_image(int(label)) for label in y])[..., None]

indices = rng.permutation(N)
X, y = X[indices], y[indices]
n_train = int(0.70 * N)
n_val = int(0.15 * N)
X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, image, label in zip(axes, X_train[:6], y_train[:6]):
    ax.imshow(image.squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'class {label}')
    ax.axis('off')
plt.tight_layout()

## 2. Baseline

Baseline навмисно простий: Flatten + один Dense softmax. Він створює контрольну точку. CNN має сенс лише якщо дає кращу generalization або іншу практичну перевагу.

In [ ]:
baseline = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SIZE, SIZE, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(3, activation='softmax')
])
baseline.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
baseline.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=12, batch_size=32, verbose=0)
baseline_test = baseline.evaluate(X_test, y_test, verbose=0)
print({'baseline_test_loss': baseline_test[0], 'baseline_test_accuracy': baseline_test[1]})

## 3. CNN candidate + early stopping

Validation використовується для early stopping. Test до завершення tuning не використовується.

In [ ]:
cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SIZE, SIZE, 1)),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(3, activation='softmax')
])
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)]
history = cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, batch_size=32,
    callbacks=callbacks, verbose=0
)
cnn.summary()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 4. Final test evaluation

Після завершення вибору model/checkpoint виконуємо final test один раз і порівнюємо з baseline.

In [ ]:
cnn_test = cnn.evaluate(X_test, y_test, verbose=0)
print({'cnn_test_loss': cnn_test[0], 'cnn_test_accuracy': cnn_test[1]})
print('Accuracy delta vs baseline:', cnn_test[1] - baseline_test[1])

## 5. Error analysis

Aggregate accuracy не показує, які класи плутаються. Будуємо confusion matrix і переглядаємо помилкові приклади.

In [ ]:
proba = cnn.predict(X_test, verbose=0)
pred = np.argmax(proba, axis=1)
cm = tf.math.confusion_matrix(y_test, pred, num_classes=3).numpy()
print('Confusion matrix:\n', cm)

wrong = np.where(pred != y_test)[0]
print('Errors:', len(wrong), 'of', len(y_test))
if len(wrong):
    fig, axes = plt.subplots(1, min(6, len(wrong)), figsize=(10, 2))
    axes = np.atleast_1d(axes)
    for ax, idx in zip(axes, wrong[:6]):
        ax.imshow(X_test[idx].squeeze(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f't={y_test[idx]} p={pred[idx]}')
        ax.axis('off')
    plt.tight_layout()

## 6. Що змінити як контрольовані експерименти

Спробуйте по одному фактору:

1. Dropout `0.0 → 0.25 → 0.5`.
2. Filters `8/16 → 16/32 → 32/64`.
3. Noise level у synthetic generator.
4. Менший train set для демонстрації overfitting.
5. Random shift range як простий domain-shift stress test.

Для кожного run запишіть hypothesis, change, validation result, compute/time і decision. Не обирайте архітектуру за final test.

## 7. Аналітичний висновок

Після виконання сформулюйте 5–7 речень:

- чи перевищив CNN baseline;
- яка evidence це підтверджує;
- які типи помилок залишилися;
- чи є ознаки overfit/domain shift;
- яке обмеження найсуттєвіше;
- який наступний experiment має найбільшу інформаційну цінність.